# Reproduce the submission from the stored weights

This notebook regenerates the final Kaggle submission **only from files in this repository**
(code in `src/`, weights in `weights/`) and the official competition test images.
No training data or internet access is needed.

Steps
1. resize the 180 test images to the network resolution(s) used by the stored models;
2. predict filament probabilities with every model listed in `weights/manifest.json`
   (test-time augmentation as recorded in the manifest, e.g. the 8 rotations/reflections "d4";
   all maps averaged);
3. turn the averaged probability map into separate filaments with the tuned post-processing;
4. write `submissions/reproduced.csv` in the competition format and check it.

**Requirements:** Python 3.12, the packages in `requirements.txt`, an NVIDIA GPU (8 GB is enough).
Place the competition data at `data/MAGFiLO_1.0_Kaggle_2026/` (as unzipped from Kaggle).

In [ ]:
import hashlib, json, sys, time
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebook started from notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import torch
from common import PRED_DIR, SUB_DIR, TEST_IMG_DIR

print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("test images:", len(list(TEST_IMG_DIR.glob("*.jpeg"))), "in", TEST_IMG_DIR)
assert torch.cuda.is_available(), "a CUDA GPU is required"
assert len(list(TEST_IMG_DIR.glob("*.jpeg"))) == 180, "competition test images not found"


## 1. Models and post-processing stored in the repository

In [ ]:
WEIGHTS = ROOT / "weights"
manifest = json.loads((WEIGHTS / "manifest.json").read_text())
runs = manifest["models"]
sizes = {}
for r in runs:
    cfg = json.loads((WEIGHTS / r / "config.json").read_text())
    sizes[r] = int(cfg.get("size", 1024))
    mb = (WEIGHTS / r / "best.pt").stat().st_size / 1e6
    print(f"{r:16s} {cfg['arch']}-{cfg['encoder']}  input {sizes[r]} px  fold {cfg['fold']}  {mb:.0f} MB (fp16)")
print("post-processing:", manifest["post_processing"])
print("cross-validated scores of this configuration:", manifest["validation"])


## 2. Resize the test images (cached under `data/cache/`)

In [ ]:
from prepare import cache_images

for size in sorted(set(sizes.values())):
    cache_images(size, include_train=False)


## 3. Predict probability maps (all models, test-time augmentation from the manifest)

In [ ]:
from infer import predict_maps, split_stems

stems = split_stems("test")
prob_dir = PRED_DIR / "reproduced" / "test"
tta = manifest.get("tta", "flip4")  # manifests written before the field existed used flip4
seconds = predict_maps(runs, stems, prob_dir, weights_root=WEIGHTS, tta=tta)
print(f"inference ({tta}): {seconds / 60:.1f} min for {len(stems)} images")


## 4. Probability maps → filament instances → submission file

In [ ]:
from make_submission import build_submission, load_params, validate

params = load_params(str(WEIGHTS / "manifest.json"))
sub = build_submission(prob_dir, stems, params)
validate(sub)
SUB_DIR.mkdir(exist_ok=True)
out = SUB_DIR / "reproduced.csv"
sub.to_csv(out, index=False)
print("wrote", out, "| sha256", hashlib.sha256(out.read_bytes()).hexdigest()[:16])
sub.head()


## 5. Compare with the file submitted to Kaggle (if present)

GPU convolution algorithms can round a few borderline pixels differently between runs,
so a handful of masks may differ slightly; the number of filaments should match.

In [ ]:
submitted = SUB_DIR / manifest.get("submitted_file", "final.csv")
if submitted.exists():
    import pandas as pd
    ref = pd.read_csv(submitted)
    same = ref.equals(sub)
    print(f"submitted: {len(ref)} rows | reproduced: {len(sub)} rows | identical: {same}")
else:
    print("submitted file not found:", submitted)
if "submitted_sha256" in manifest:  # fingerprint stored at export time
    same_hash = hashlib.sha256(out.read_bytes()).hexdigest() == manifest["submitted_sha256"]
    print("sha256 identical to the submitted file:", same_hash, "| rows:", manifest.get("submitted_rows"))


## 6. Look at a prediction

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from pycocotools import mask as mask_utils

stem = stems[0]
img = cv2.imread(str(TEST_IMG_DIR / f"{stem}.jpeg"), cv2.IMREAD_GRAYSCALE)
vis = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
for rle in sub.loc[sub.filament_id.str.startswith(stem + "_"), "segmentation_rle"]:
    m = mask_utils.decode({"size": [2048, 2048], "counts": rle.encode("ascii")})
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(vis, contours, -1, (255, 60, 60), 3)
plt.figure(figsize=(8, 8)); plt.imshow(vis); plt.axis("off"); plt.title(stem); plt.show()
